# YUVA Internship – Week 2
## Data Collection, Cleaning, and Preprocessing for Logistics Analysis

This notebook implements the Week 2 preprocessing pipeline described in the report.


### Objectives
- Simulate collection of a public logistics dataset.
- Profile missing values, duplicates, data types, and ranges.
- Clean dates and categorical fields.
- Detect potential outliers using the IQR method.
- Create order-level logistics features.
- Demonstrate Min-Max and StandardScaler preprocessing.
- Validate the cleaned dataset.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.week2_cleaning import (
    load_csv, profile_dataframe, clean_orders,
    clean_products, iqr_bounds,
    aggregate_order_items, build_clean_analysis_dataset
)
from src.preprocessing import minmax_scale, standardize


In [ ]:
orders = load_csv('olist_orders_dataset.csv')
items = load_csv('olist_order_items_dataset.csv')
products = load_csv('olist_products_dataset.csv')

print('Orders:', orders.shape)
print('Items:', items.shape)
print('Products:', products.shape)


In [ ]:
orders_profile = profile_dataframe(orders)
orders_profile.head(15)


In [ ]:
print('Exact duplicate order rows:', orders.duplicated().sum())
print('Duplicate order IDs:', orders['order_id'].duplicated().sum())

print('\nMissing values:')
print(orders.isna().sum().sort_values(ascending=False).head(10))


In [ ]:
orders_clean = clean_orders(orders)
products_clean = clean_products(products)

orders_clean[['order_id', 'delivery_days', 'late_flag']].head()


In [ ]:
analysis_df = build_clean_analysis_dataset(orders, items)
analysis_df[['order_id', 'delivery_days', 'late_flag', 'order_value', 'freight_value', 'item_count']].head()


## Outlier Detection – IQR

Potential outliers are screened rather than automatically deleted because extreme logistics events may be genuine.


In [ ]:
freight = items['freight_value'].dropna()
lower, upper = iqr_bounds(freight)
outlier_mask = (freight < lower) | (freight > upper)

print('IQR lower bound:', round(lower, 2))
print('IQR upper bound:', round(upper, 2))
print('Potential freight outliers:', int(outlier_mask.sum()))


## Scaling

Scaling is useful for downstream algorithms such as K-Means. In a real predictive pipeline, scalers should be fitted only on training data.


In [ ]:
scale_df = analysis_df[['order_value', 'freight_value', 'item_count']].dropna().copy()

minmax_df, minmax_scaler = minmax_scale(
    scale_df,
    ['order_value', 'freight_value', 'item_count']
)

standard_df, standard_scaler = standardize(
    scale_df,
    ['order_value', 'freight_value', 'item_count']
)

print('Min-Max sample:')
display(minmax_df.head())
print('Standardized sample:')
display(standard_df.head())


In [ ]:
print('Rows before cleaning:', len(orders))
print('Rows after exact-duplicate removal:', len(orders_clean))
print('Negative delivery days:', int((analysis_df['delivery_days'] < 0).sum()))
print('Missing delivery days:', int(analysis_df['delivery_days'].isna().sum()))
print('\nLate flag counts:')
print(analysis_df['late_flag'].value_counts(dropna=False))


## Week 2 Reflection

The cleaned dataset is now suitable as a foundation for the KPI analysis, predictive modelling, clustering, and route-optimization work planned in later stages. The next step is to expand the feature set with seller, customer, product, review, and geographic information while preserving data lineage and preventing leakage.
